In [ ]:
# =========================================================================
# PORTFOLIO PLOT — performance charts for the SELECTED allocation only
# =========================================================================
# Companion to plot.ipynb, but scoped to whatever `portfolio_use` selects
# (AI wave allocation by default, or the sector/satellite allocation). It
# plots ONLY the stocks/ETFs that live in that allocation, across a set of
# timeframes, and adds a Watchlist tab for the bench names.
#
# Two main tabs:
#   * Baskets   — sector mode: ETFs + each satellite (Nuclear, Quantum, ...)
#                 ai mode    : W1 Silicon ... W6 Speculative
#   * Watchlist — the not-held candidates, grouped by their strategy tag
#
# Every chart is available for each value in `timeframes` (years).
#
# Switch allocation / timeframes WITHOUT editing this notebook from a driver:
#
#     timeframes   = [0.1, 0.5, 1, 2, 5, 10]
#     portfolio_use = 'ai_allocation'    # or 'allocation' for the sector book
#     run_notebook('portfolio_plot.ipynb')
# =========================================================================

import os
import sys

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Stocks'
except ImportError:
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import re
import time
import base64
import logging
import warnings
from io import BytesIO
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FixedLocator, FixedFormatter
import IPython

# Silence yfinance's internal logger and noisy warnings so only genuine
# errors raised by this notebook surface.
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

# =========================================================================
# 0. RESOLVE TIMEFRAMES + WHICH ALLOCATION TO PLOT
# =========================================================================
# `timeframes` (years) and `portfolio_use` may be injected by a driver cell;
# otherwise fall back to sensible defaults / the PORTFOLIO_USE env var.
try:
    timeframes  # noqa: F821  (may be injected by a driver cell)
except NameError:
    timeframes = [0.1, 0.5, 1, 2, 5, 10]
timeframes = list(timeframes)

try:
    _sel_raw = portfolio_use  # noqa: F821  (may be injected by a driver cell)
except NameError:
    _sel_raw = os.environ.get('PORTFOLIO_USE', 'ai')
_sel = str(_sel_raw).strip().lower()
_AI_ALIASES = {'ai', 'ai_allocation', 'ai_allocations', 'wave', 'waves'}
_SEC_ALIASES = {'allocation', 'allocations', 'sector', 'satellite', 'sectors'}
if _sel in _AI_ALIASES:
    ALLOCATION_MODE = 'ai'
elif _sel in _SEC_ALIASES:
    ALLOCATION_MODE = 'sector'
else:
    warnings.warn(f"Unrecognized portfolio_use={_sel_raw!r}; defaulting to AI allocation.")
    ALLOCATION_MODE = 'ai'

# =========================================================================
# 0b. BUILD A UNIFIED VIEW OF THE SELECTED ALLOCATION
# =========================================================================
# Normalize both allocations into the same shapes so the rendering code below
# is structure-agnostic:
#   BASKET_SLICES : ordered [(slice_id, label, [tickers])]   (Baskets tab)
#   WATCH_GROUPS  : ordered [(group_id, label, [tickers])]   (Watchlist tab)

if ALLOCATION_MODE == 'sector':
    from portfolio.allocations import (
        TARGET_WEIGHTS,
        NUCLEAR_BASKET_TARGETS, QUANTUM_BASKET_TARGETS, CYBER_BASKET_TARGETS,
        INDUSTRIAL_BASKET_TARGETS, SPECGROWTH_BASKET_TARGETS, OTHER_BASKET_TARGETS,
        WATCHLIST_EXCLUDED,
    )
    ALLOC_TITLE = 'Portfolio Plot — ETF/Satellite'
    _ETF_TICKERS = ['XAIX.DE', 'SMHV.SW', 'QDVE.DE']
    # ETFs grouped into one slice, then each satellite basket as its own slice.
    BASKET_SLICES = [
        ('ETFs',        'ETFs',   list(_ETF_TICKERS)),
        ('NUCLEAR',     'NUC',    [t for t in NUCLEAR_BASKET_TARGETS]),
        ('QUANTUM',     'QTM',    [t for t in QUANTUM_BASKET_TARGETS]),
        ('CYBER',       'CYBER',  [t for t in CYBER_BASKET_TARGETS]),
        ('INDUSTRIAL',  'IND',    [t for t in INDUSTRIAL_BASKET_TARGETS]),
        ('SPECGROWTH',  'SPEC',   [t for t in SPECGROWTH_BASKET_TARGETS]),
        ('OTHER',       'OTHER',  [t for t in OTHER_BASKET_TARGETS]),
    ]
    # Sector bench uses the strategy vocabulary cycle / lottery_ticket.
    _WL = WATCHLIST_EXCLUDED
    _WL_ORDER = [
        ('cycle',          'Cycle'),
        ('lottery_ticket', 'Lottery'),
    ]

else:  # 'ai' — AI value-chain wave allocation
    from portfolio.AI_allocations import (
        W1_SILICON_TARGETS, W2_POWER_TARGETS, W3_DCINFRA_TARGETS,
        W4_CLOUD_TARGETS, W5_SOFTWARE_TARGETS, W6_SPEC_TARGETS,
        WATCHLIST,
    )
    ALLOC_TITLE = 'Portfolio Plot — AI Waves'
    # AI allocation has no directly-held ETF slice (SMHV.SW lives inside W1).
    BASKET_SLICES = [
        ('W1_SILICON',  'W1',  [t for t in W1_SILICON_TARGETS]),
        ('W2_POWER',    'W2',  [t for t in W2_POWER_TARGETS]),
        ('W3_DCINFRA',  'W3',  [t for t in W3_DCINFRA_TARGETS]),
        ('W4_CLOUD',    'W4',  [t for t in W4_CLOUD_TARGETS]),
        ('W5_SOFTWARE', 'W5',  [t for t in W5_SOFTWARE_TARGETS]),
        ('W6_SPEC',     'W6',  [t for t in W6_SPEC_TARGETS]),
    ]
    _WL = WATCHLIST
    _WL_ORDER = [
        ('dca',      'DCA'),
        ('cycle',    'Cycle'),
        ('catalyst', 'Catalyst'),
        ('lottery',  'Lottery'),
    ]

# Build the watchlist groups (skip empty strategy buckets).
WATCH_GROUPS = []
for _strat, _label in _WL_ORDER:
    _names = sorted(t for t, d in _WL.items() if d.get('strategy') == _strat)
    if _names:
        WATCH_GROUPS.append((_strat, _label, _names))

# =========================================================================
# 0c. COLOR ASSIGNMENT (deterministic per ticker)
# =========================================================================
_PALETTE = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b',
    '#e377c2', '#17becf', '#bcbd22', '#7f7f7f', '#393b79', '#637939',
    '#8c6d31', '#843c39', '#7b4173', '#3182bd', '#e6550d', '#31a354',
    '#756bb1', '#636363', '#ad494a', '#a55194', '#6baed6', '#fd8d3c',
]
_LINESTYLES = ['-', '--', '-.', ':']
_color_cache = {}

def style_for(ticker):
    if ticker not in _color_cache:
        idx = len(_color_cache)
        _color_cache[ticker] = {
            'color': _PALETTE[idx % len(_PALETTE)],
            'linestyle': _LINESTYLES[(idx // len(_PALETTE)) % len(_LINESTYLES)],
        }
    return _color_cache[ticker]

# =========================================================================
# 1. FETCH PRICE HISTORY for every ticker we intend to plot
# =========================================================================
_all_tickers = set()
for _sid, _lbl, _tks in BASKET_SLICES:
    _all_tickers.update(_tks)
for _gid, _lbl, _tks in WATCH_GROUPS:
    _all_tickers.update(_tks)
_all_tickers = sorted(_all_tickers)

plot_data = {}
earliest_dates = {}


def _fetch_history(tickers, retries=3, delay=1.0):
    """Bulk-download max history; resilient to transient yfinance failures."""
    last_err = None
    for attempt in range(retries):
        try:
            bulk = yf.download(
                tickers=tickers, period='max', group_by='ticker',
                auto_adjust=True, threads=True, progress=False,
            )
            return bulk
        except Exception as exc:  # noqa: BLE001
            last_err = exc
            time.sleep(delay * (attempt + 1))
    warnings.warn(f"Price download failed after {retries} tries: {last_err}")
    return None


if _all_tickers:
    _bulk = _fetch_history(_all_tickers)
    if _bulk is not None and not _bulk.empty:
        for t in _all_tickers:
            try:
                if len(_all_tickers) > 1:
                    df_asset = _bulk[t].copy()
                else:
                    df_asset = _bulk.copy()
                df_asset.index = pd.to_datetime(df_asset.index).tz_localize(None)
                df_clean = df_asset.dropna(subset=['Close'])
                if not df_clean.empty:
                    df_clean = df_clean.sort_index()
                    plot_data[t] = df_clean
                    earliest_dates[t] = df_clean.index.min()
            except (KeyError, TypeError):
                continue

# =========================================================================
# 2. CHART + HTML HELPERS
# =========================================================================
html_elements = [
    '<html><head><style>',
    'body { font-family: Arial, sans-serif; margin: 20px; background-color: #f8f9fa; color: black; }',
    '.po-title { font-size: 22px; font-weight: bold; margin: 4px 0 14px 0; }',
    '.report-section { margin-bottom: 30px; background: white; padding: 16px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); overflow-x: auto; }',
    'img { max-width: 100%; height: auto; display: block; margin: 10px 0; }',
    '.tab { overflow: hidden; border: 1px solid #000; background-color: #f1f1f1; margin-top: 12px; }',
    '.tab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 12px 16px; transition: 0.3s; color: black; font-weight: bold; }',
    '.tab button:hover { background-color: #ddd; }',
    '.tab button.active { background-color: #ccc; }',
    '.tabcontent { display: none; padding: 16px; border: 1px solid #000; border-top: none; background: white; }',
    '.subtab { overflow: hidden; background-color: #e9e9e9; border: 1px solid #000; border-bottom: none; }',
    '.subtab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 9px 13px; transition: 0.3s; color: black; font-size: 14px; }',
    '.subtab button:hover { background-color: #d5d5d5; }',
    '.subtab button.active { background-color: #bbb; }',
    '.timetab { overflow: hidden; background-color: #f2f2f2; border: 1px solid #000; border-top: none; border-bottom: none; }',
    '.timetab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 7px 12px; transition: 0.3s; color: black; font-size: 13px; }',
    '.timetab button:hover { background-color: #e0e0e0; }',
    '.timetab button.active { background-color: #cfcfcf; }',
    '</style>',
    '<script>',
    'function openTab(evt, tabName, tabClass, contentClass) {',
    '  var i, tabcontent, tablinks;',
    '  tabcontent = document.getElementsByClassName(contentClass);',
    "  for (i = 0; i < tabcontent.length; i++) { tabcontent[i].style.display = 'none'; }",
    '  tablinks = document.getElementsByClassName(tabClass);',
    "  for (i = 0; i < tablinks.length; i++) { tablinks[i].className = tablinks[i].className.replace(' active', ''); }",
    "  document.getElementById(tabName).style.display = 'block';",
    "  if (evt) { evt.currentTarget.className += ' active'; }",
    '}',
    "document.addEventListener('keydown', function(e) {",
    "  if (e.key === 'Tab') {",
    '    e.preventDefault();',
    "    var visibleMain = document.querySelector('.main-tabcontent[style*=\"display: block\"]');",
    '    if (!visibleMain) { return; }',
    "    var subBtns = visibleMain.querySelectorAll('.subtab button');",
    '    if (!subBtns.length) { return; }',
    '    var activeIdx = -1;',
    '    for (var i = 0; i < subBtns.length; i++) {',
    "      if (subBtns[i].className.indexOf('active') !== -1) { activeIdx = i; break; }",
    '    }',
    '    var nextBtn = subBtns[(activeIdx + 1) % subBtns.length];',
    '    nextBtn.click();',
    '  }',
    '});',
    '</script>',
    '</head><body>',
    f'<div class="po-title">{ALLOC_TITLE}</div>',
]


def apply_y_scaling(ax):
    ymin, ymax = ax.get_ylim()
    span = ymax - ymin
    raw_step = span / 8.0
    step = max(10, int(np.ceil(raw_step / 10.0)) * 10)
    new_ymin = np.floor(ymin / step) * step
    new_ymax = np.ceil(ymax / step) * step
    if new_ymin == new_ymax:
        new_ymax += step
        new_ymin -= step
    ticks = np.arange(new_ymin, new_ymax + step, step)
    ax.set_ylim(new_ymin, new_ymax)
    ax.set_yticks(ticks)
    ax.set_yticklabels([str(int(t)) for t in ticks])


def handle_output(fig, element_id):
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=150)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode('utf-8')
    buf.close()
    plt.close(fig)
    html_elements.append(f'<img id="{element_id}" src="data:image/png;base64,{b64}"/>')


def render_chart(tickers, title_base, years, element_id):
    """Render one normalized-performance (%) chart for a set of tickers."""
    fig, ax = plt.subplots(figsize=(10.8, 5.4))
    start_date = datetime.now() - timedelta(days=int(years * 365))
    vlines_to_draw = []
    plotted = 0
    for name in tickers:
        if name not in plot_data or plot_data[name].empty:
            continue
        df = plot_data[name]
        df = df[df.index >= start_date].copy()
        if df.empty:
            continue
        base = df['Close'].iloc[0]
        df['Pct'] = ((df['Close'] - base) / base) * 100
        st = style_for(name)
        ax.plot(df.index, df['Pct'], label=name, color=st['color'],
                linestyle=st['linestyle'], linewidth=1.6)
        plotted += 1
        true_start = earliest_dates.get(name, start_date)
        if true_start > start_date:
            vlines_to_draw.append((true_start, st['color']))
    if plotted:
        ymin_val, _ = ax.get_ylim()
        for birth, col in vlines_to_draw:
            ax.vlines(x=birth, ymin=ymin_val, ymax=0, colors=col,
                      linestyles='--', alpha=0.7, linewidth=1.1)
    else:
        ax.text(0.5, 0.5, 'No price data', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='#888')
    ax.set_title(f'{title_base} ({years}y)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Performance (%)', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.axhline(0, color='red', linestyle='-', alpha=0.4)
    ax.set_xlim(left=start_date, right=datetime.now())
    if plotted:
        apply_y_scaling(ax)
    normalized_start = start_date.replace(month=1, day=1)
    extended_end = datetime.now() + timedelta(days=31)
    tick_dates = pd.date_range(start=normalized_start, end=extended_end, freq='MS')
    tick_locs = [mdates.date2num(d) for d in tick_dates]
    sparse_labels = []
    for d in tick_dates:
        if years <= 0.5:
            sparse_labels.append(d.strftime('%b %Y').upper())
        elif d.month == 1:
            sparse_labels.append(d.strftime('JAN %Y'))
        elif d.month == 6:
            sparse_labels.append(d.strftime('JUNE %Y'))
        else:
            sparse_labels.append('')
    ax.xaxis.set_major_locator(FixedLocator(tick_locs))
    ax.xaxis.set_major_formatter(FixedFormatter(sparse_labels))
    ax.tick_params(axis='x', rotation=35, labelsize=9)
    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
    handle_output(fig, element_id)


def render_slice(slice_id, label, tickers, tab_class, content_class):
    """A slice panel: timeframe sub-buttons, one chart per timeframe."""
    html_elements.append(f'<div id="{slice_id}" class="{content_class}">')
    html_elements.append('<div class="timetab">')
    for idx, y in enumerate(timeframes):
        def_id = f' id="default_{slice_id}_time"' if idx == 0 else ''
        html_elements.append(
            f'<button class="{slice_id}-time-links" onclick="openTab(event, '
            f"'{slice_id}_{y}y', '{slice_id}-time-links', '{slice_id}-time-content')\""
            f'{def_id}>{y}y</button>'
        )
    html_elements.append('</div>')
    for y in timeframes:
        html_elements.append(f'<div id="{slice_id}_{y}y" class="{slice_id}-time-content">')
        render_chart(tickers, label, y, f'{slice_id.lower()}_{y}y_img')
        html_elements.append('</div>')
    html_elements.append('</div>')


# =========================================================================
# 3. TOP-LEVEL TABS
# =========================================================================
html_elements.append('<div class="tab">')
html_elements.append(
    '<button class="main-tab-links" onclick="openTab(event, \'Baskets\', '
    "'main-tab-links', 'main-tabcontent')\" id="
    '"defaultMainTab">Baskets</button>'
)
html_elements.append(
    '<button class="main-tab-links" onclick="openTab(event, \'Watchlist\', '
    "'main-tab-links', 'main-tabcontent')\">Watchlist</button>"
)
html_elements.append('</div>')

_default_clicks = ['defaultMainTab']

# --- TAB 1: BASKETS ---------------------------------------------------------
html_elements.append('<div id="Baskets" class="main-tabcontent">')
html_elements.append('<div class="subtab">')
for idx, (sid, label, tks) in enumerate(BASKET_SLICES):
    def_id = ' id="defaultBasketSub"' if idx == 0 else ''
    html_elements.append(
        f'<button class="basket-sub-links" onclick="openTab(event, '
        f"'B_{sid}', 'basket-sub-links', 'basket-subcontent')\"{def_id}>{label}</button>"
    )
html_elements.append('</div>')
_default_clicks.append('defaultBasketSub')
for sid, label, tks in BASKET_SLICES:
    render_slice(f'B_{sid}', label, tks, 'basket-sub-links', 'basket-subcontent')
    _default_clicks.append(f'default_B_{sid}_time')
html_elements.append('</div>')

# --- TAB 2: WATCHLIST -------------------------------------------------------
html_elements.append('<div id="Watchlist" class="main-tabcontent">')
if WATCH_GROUPS:
    html_elements.append('<div class="subtab">')
    for idx, (gid, label, tks) in enumerate(WATCH_GROUPS):
        def_id = ' id="defaultWatchSub"' if idx == 0 else ''
        html_elements.append(
            f'<button class="watch-sub-links" onclick="openTab(event, '
            f"'W_{gid}', 'watch-sub-links', 'watch-subcontent')\"{def_id}>{label}</button>"
        )
    html_elements.append('</div>')
    _default_clicks.append('defaultWatchSub')
    for gid, label, tks in WATCH_GROUPS:
        render_slice(f'W_{gid}', f'Watchlist — {label}', tks,
                     'watch-sub-links', 'watch-subcontent')
        _default_clicks.append(f'default_W_{gid}_time')
else:
    html_elements.append('<div class="report-section">No watchlist names for this allocation.</div>')
html_elements.append('</div>')

# =========================================================================
# 4. DEFAULT-TAB ACTIVATION + DISPLAY
# =========================================================================
_clicks_js = '\n'.join(
    f"  var b = document.getElementById('{cid}'); if (b) b.click();"
    for cid in _default_clicks
)
html_elements.append('<script>\n' + _clicks_js + '\n</script>')
html_elements.append('</body></html>')

html_content = '\n'.join(html_elements)
IPython.display.display(IPython.display.HTML(html_content))
